# Requirement 5 · Optimizers and ablations

Batch GD, SGD (batch 1), mini-batch SGD, Momentum and Nesterov, then one-change-at-a-time ablations of
initialisation, normalisation, regularisation and augmentation, on real face frames (engaged / neutral / disengaged).
See `docs/guide/05_optimizers_ablations.md`.

**CPU runtime is enough** (about 25 minutes; most of it is SGD with batch size 1, which makes 1,293 updates per epoch).

## 0 · Put the project on Google Drive (once)

1. Zip the `smart Attendance` folder on your laptop **without** `data/`, `models/` and `reports/`
   (they are rebuilt here). Rename the zip to `smart_attendance.zip`.
2. Upload it to `MyDrive/` in Google Drive.

Keeping data and results on Drive means they survive when Colab disconnects (Colab's own disk is wiped).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
PROJECT = '/content/drive/MyDrive/smart_attendance'
if not os.path.exists(PROJECT):
    zipfile.ZipFile('/content/drive/MyDrive/smart_attendance.zip').extractall(PROJECT)
# if the zip contained a top-level folder, step into it
inner = [d for d in os.listdir(PROJECT) if os.path.isfile(os.path.join(PROJECT, d, 'pyproject.toml'))]
if inner:
    PROJECT = os.path.join(PROJECT, inner[0])
%cd "$PROJECT"
!ls

In [ ]:
# Colab already has torch, torchvision and OpenCV; this adds the rest and installs our package
!pip install -q -r requirements.txt ultralytics kagglehub
!pip install -q -e .

In [ ]:
# the drowsiness face frames (Hugging Face, MIT licence), downloaded by the data stage
import os
if not os.path.exists('data/raw/drowsiness'):
    !python -m smartattend stage data

## 1 · Run everything

In [ ]:
!python -m smartattend optimize

In [ ]:
from IPython.display import Image, display
for name in ['optimizer_paths', 'optimizer_comparison_real', 'ablations_real', 'overfitting_real']:
    display(Image(f'reports/figures/nnfund/{name}.png'))

## 2 · Try one change yourself

In [ ]:
from smartattend.nnfund import optimization as opt
data = opt.load_faces('data/raw/drowsiness')
for change in [{}, {'init': 'zeros'}, {'aug': 'shift'}, {'dropout': 0.3, 'l2': 1e-3}]:
    h = opt.train(data, epochs=30, seed=0, **{**opt.BASE, **change})
    print(f"{str(change):35s} train F1 {h['train_f1'][-1]:.3f}  val F1 {h['val_f1_last']:.3f}  test F1 {h['test_f1_last']:.3f}")

## 3 · Tests

In [ ]:
!python -m pytest -q tests/test_optimization.py